과제 수행 시 활용한 생성형 AI 도구 대화 기록: https://chatgpt.com/share/69fcbcd8-ade8-8323-8e48-b80b695861c4

# Q1: 학생 성적 보고서 생성기

In [1]:
import csv, json, logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    )

def make_report(csv_path: str, json_path: str) -> int:
    students = []

    try:
        with open(csv_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)

            for row in reader:
                name = row["이름"]
                student_id = row["학번"]
                mid = row["중간"]
                fin = row["기말"]
                assg = row["과제"]

                # 결측치가 없는 경우
                if mid and fin and assg:
                    mid = int(mid)
                    fin = int(fin)
                    assg = int(assg)

                    avg = mid * 0.3 + fin * 0.5 + assg * 0.2

                    if avg >= 90:
                        grade = "A"
                    elif avg >= 80:
                        grade = "B"
                    elif avg >= 70:
                        grade = "C"
                    else:
                        grade = "F"
                
                # 결측치가 있는 경우
                else:
                    mid = int(mid) if mid else None
                    fin = int(fin) if fin else None
                    assg = int(assg) if assg else None

                    avg = None
                    grade = None
                
                
                score_data = {
                    "중간": mid,
                    "기말": fin,
                    "과제": assg
                }

                student = {
                    "이름": name,
                    "학번": student_id,
                    "점수": score_data,
                    "평균": avg,
                    "등급": grade
                }

                students.append(student)

                logging.info(f"{name}: {avg}, {grade}")
        
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(students, f, ensure_ascii=False, indent=2)

        return len(students)
    
    except FileNotFoundError:
        logging.warning("CSV 파일을 찾을 수 없습니다.")
        return 0
    
    except UnicodeDecodeError:
        logging.error("CSV 파일 인코딩 오류가 발생했습니다.")
        return 0

In [2]:
# 함수 호출
make_report("scores.csv", "report.json")

2026-05-07 16:23:10,654 [INFO] 김언어: 89.5, B
2026-05-07 16:23:10,657 [INFO] 이국문: 84.4, B
2026-05-07 16:23:10,658 [INFO] 박영문: 93.5, A
2026-05-07 16:23:10,659 [INFO] 최역사: None, None


4

In [3]:
# 생성된 report.json의 전체 내용

with open("report.json", "r", encoding="utf-8") as f:
    for i in f:
        print(i, end='')

[
  {
    "이름": "김언어",
    "학번": "2026-10000",
    "점수": {
      "중간": 85,
      "기말": 92,
      "과제": 90
    },
    "평균": 89.5,
    "등급": "B"
  },
  {
    "이름": "이국문",
    "학번": "2026-12345",
    "점수": {
      "중간": 78,
      "기말": 88,
      "과제": 85
    },
    "평균": 84.4,
    "등급": "B"
  },
  {
    "이름": "박영문",
    "학번": "2026-13579",
    "점수": {
      "중간": 95,
      "기말": 90,
      "과제": 100
    },
    "평균": 93.5,
    "등급": "A"
  },
  {
    "이름": "최역사",
    "학번": "2025-11111",
    "점수": {
      "중간": null,
      "기말": 82,
      "과제": 88
    },
    "평균": null,
    "등급": null
  }
]

설명: 중간, 기말, 과제 중 하나라도 값이 존재하지 않으면 해당 값을 None으로 대체하고 평균과 등급까지 None으로 처리하였다. 또한 한글 텍스트의 표준인 utf-8로 인코딩했다.

#

# Q2: 사용자 정의 예외와 자모 분류

In [4]:
# (a)

class InvalidJamoError(ValueError):
    """한글 자모가 아닌 문자"""

In [5]:
# (b)

def classify_jamo(c: str) -> str:

    if not isinstance(c, str):
        raise TypeError("입력은 str 타입이어야 합니다.")
    
    if len(c) != 1:
        raise ValueError("길이가 1인 문자열만 입력 가능합니다.")

    code = ord(c)

    if 0x3131 <= code <= 0x314E:
        return "자음"
    elif 0x314F <= code <= 0x3163:
        return "모음"
    else:
        raise InvalidJamoError(f"유효한 한글 자모가 아닙니다: {c}")

In [6]:
# (c)

inputs = ["ㄱ", "ㅏ", "ㄲ", "가", "AB", 5, "ㅎ", "ㅣ", ""]

for i in inputs:

    try:
        jamo = classify_jamo(i)
        print(f"{i}: {jamo}")
    
    except InvalidJamoError as e:
        print(f"[{type(e).__name__}] {e}")
    
    except ValueError as e:
        print(f"[{type(e).__name__}] {e}")

    except TypeError as e:
        print(f"[{type(e).__name__}] {e}")


ㄱ: 자음
ㅏ: 모음
ㄲ: 자음
[InvalidJamoError] 유효한 한글 자모가 아닙니다: 가
[ValueError] 길이가 1인 문자열만 입력 가능합니다.
[TypeError] 입력은 str 타입이어야 합니다.
ㅎ: 자음
ㅣ: 모음
[ValueError] 길이가 1인 문자열만 입력 가능합니다.


설명: InvalidJamoError의 경우 입력값의 타입은 str으로 적절하지만, 값이 자음이나 모음에 해당하지 않는 것이다. 따라서 InvalidJamoError를 Exception보다는 ValueError의 자식 클래스로 만드는 것이 더 적절하다. 실제 코드에서는 InvalidJamoError가 다른 ValueError와는 별개로 처리될 수 있도록 InvalidJamoError에 대한 except절을 ValueError에 대한 except절보다 먼저 적었다.